# Q2.3 LoRA Adapters

### What this notebook does
1. Hyperparameter tuning finds best r, lr, and confirms if class weights are needed
2. Trains 3 LoRA adapters (en-UK, en-AU, en-IN) with best params
3. Evaluates each adapter on all 3 test sets  3×3 cross-variety matrix
4. Visualises results and uploads adapters to HuggingFace Hub


## 1. Setup

In [ ]:
# Cell 1 — GPU check
import torch
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: No GPU")

In [ ]:
# Cell 2 — clone repo and set working directory
import os, sys

if not os.path.exists("NLP-sequence-classification"):
    !git clone https://github.com/momofahmi/NLP-sequence-classification.git

os.chdir("NLP-sequence-classification")
sys.path.append(os.path.abspath("."))
print("Working directory:", os.getcwd())
print("Contents:", os.listdir("."))

In [ ]:
# Cell 3 — install dependencies
!pip install -r requirements.txt -q
print("Dependencies installed")

In [ ]:
# Cell 4 — authenticate HuggingFace 
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))
print("Authenticated")

In [ ]:
# Cell 5 — imports
import torch
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from datasets import load_dataset
from sklearn.metrics import (
    f1_score, classification_report,
    confusion_matrix, precision_score, recall_score,
)
from peft import PeftModel

from models.lora_adapters import (
    load_model, apply_lora, tokenize_dataset,
    training_args, save_adapter, push_adapter_to_hub,
    LoRAConfig, VARIETIES, HF_USERNAME,
)
from src.functions_to_use import class_weights, new_weighted_class

print("All imports successful")

## 2. Configuration

In [ ]:
# Cell 6 — experiment config

MODEL_KEY   = "llama-3b"   
TASK        = "Sarcasm"    
MAX_LENGTH  = 128          
SEEDS       = [42, 123]    
ADAPTER_DIR = "./adapters"
RESULTS_DIR = "./results"

os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Model     : {MODEL_KEY}")
print(f"Task      : {TASK}")
print(f"Varieties : {VARIETIES}")
print(f"Seeds     : {SEEDS}")

## 3. Load Dataset

In [ ]:
# Cell 7 — load BESSTIE and inspect sarcasm imbalance

ds = load_dataset("surrey-nlp/BESSTIE-CW-26")
print(ds)

for variety in VARIETIES:
    split  = ds["train"].filter(lambda x: x["variety"] == variety)
    n_sarc = sum(int(x) for x in split["Sarcasm"])
    n_not  = len(split) - n_sarc
    pct    = n_sarc / len(split) * 100
    print(f"  {variety}: {len(split)} examples | "
          f"sarcastic={n_sarc} ({pct:.1f}%) | "
          f"not_sarcastic={n_not} ({100-pct:.1f}%)")

## 4. Load Base Model

In [ ]:
# Cell 8 — load LLaMA once
base_model, tokenizer = load_model(MODEL_KEY)
print("\nBase model loaded and frozen. Ready for LoRA adapters.")

## 5. Hyperparameter Tuning

Before running the full 6-run training loop, we will find the best hyperparameters.

**What we tune:**
- `r` (LoRA rank): controls adapter capacity
- `lr` (learning rate): controls how fast the adapter learns
- `use_weights` (class weights): shows the impact of handling class imbalance

**Setup:** 1 epoch only, en-UK variety only, seed=42 .it will be fast and representative.

In [ ]:
# Cell 9 — hyperparameter tuning grid

from transformers import Trainer as StandardTrainer

# parameters to explore
param_grid = {
    "r"          : [4, 8],
    "lr"         : [1e-4, 2e-4],
    "use_weights": [True, False],
}

TUNE_VARIETY = "en-UK"  
tune_results = []

# prepare data once
tune_train = ds["train"].filter(lambda x: x["variety"] == TUNE_VARIETY)
tune_val   = ds["validation"].filter(lambda x: x["variety"] == TUNE_VARIETY)

# compute class weights for this variety
weights_uk = class_weights(tune_train, TASK)

print(f"Tuning on {TUNE_VARIETY} | {len(tune_train)} train | {len(tune_val)} val")
print(f"Total combinations: {len(list(itertools.product(*param_grid.values())))}\n")

for r, lr, use_w in itertools.product(
    param_grid["r"],
    param_grid["lr"],
    param_grid["use_weights"],
):
    label = f"r={r} lr={lr} weights={'yes' if use_w else 'no'}"
    print(f"Testing {label}...")

    config = LoRAConfig(r=r, lora_alpha=r*2)
    model  = apply_lora(base_model, config)

    train_tok = tokenize_dataset(tune_train, tokenizer, TASK, MAX_LENGTH)
    val_tok   = tokenize_dataset(tune_val,   tokenizer, TASK, MAX_LENGTH)

    args = training_args(
        output_dir = f"./tuning/{label.replace(' ','_')}",
        variety    = TUNE_VARIETY,
        seed       = 42,
        epochs     = 1,    # 1 epoch only for speed
        batch_size = 8,
        lr         = lr,
    )

    # use weighted or standard trainer based on config
    if use_w:
        Trainer = new_weighted_class(weights_uk)
    else:
        Trainer = StandardTrainer

    trainer = Trainer(
        model         = model,
        args          = args,
        train_dataset = train_tok,
        eval_dataset  = val_tok,
    )
    trainer.train()

    # evaluate on validation set
    preds  = trainer.predict(val_tok)
    y_pred = preds.predictions.argmax(-1)
    y_true = [int(x) for x in tune_val[TASK]]

    macro_f1  = f1_score(y_true, y_pred, average="macro")
    f1_sarc   = f1_score(y_true, y_pred, average=None)[1]
    only_one  = len(set(y_pred.tolist())) == 1

    tune_results.append({
        "r"              : r,
        "lr"             : lr,
        "use_weights"    : use_w,
        "macro_f1"       : round(macro_f1, 4),
        "f1_sarcasm"     : round(f1_sarc,  4),
        "predicts_one_class": only_one,
    })
    print(f"  Macro-F1={macro_f1:.4f} | Sarcasm-F1={f1_sarc:.4f}"
          + (" ← WARNING: predicts only one class" if only_one else ""))

print("\nTuning complete")

In [ ]:
# Cell 10 — display tuning results and pick best config

tune_df = pd.DataFrame(tune_results).sort_values("macro_f1", ascending=False)

print("=== Hyperparameter Tuning Results ===")
print(tune_df.to_string(index=False))

# best config = highest macro_f1 WITH use_weights=True
best = tune_df[tune_df["use_weights"] == True].iloc[0]

BEST_R  = int(best["r"])
BEST_LR = float(best["lr"])
print(f"\nBest config: r={BEST_R}, lr={BEST_LR}, use_weights=True")
print(f"Best Macro-F1: {best['macro_f1']:.4f}")

In [ ]:
# Cell 11 — visualise tuning results

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# plot 1 — macro F1 by config
configs = [f"r={r['r']}\nlr={r['lr']}\nw={'yes' if r['use_weights'] else 'no'}"
           for _, r in tune_df.iterrows()]
colors  = ["#4A6CF7" if r["use_weights"] else "#E24B4A"
           for _, r in tune_df.iterrows()]

axes[0].bar(range(len(tune_df)), tune_df["macro_f1"], color=colors)
axes[0].set_xticks(range(len(tune_df)))
axes[0].set_xticklabels(configs, fontsize=8)
axes[0].set_ylabel("Macro-F1")
axes[0].set_title("Macro-F1 by Config\n(blue=weighted, red=unweighted)")
axes[0].axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="random baseline")
axes[0].legend()

# plot 2 — sarcasm class F1 (most important metric)
axes[1].bar(range(len(tune_df)), tune_df["f1_sarcasm"], color=colors)
axes[1].set_xticks(range(len(tune_df)))
axes[1].set_xticklabels(configs, fontsize=8)
axes[1].set_ylabel("Sarcasm Class F1")
axes[1].set_title("Sarcasm Class F1 by Config\n(shows if model actually detects sarcasm)")
axes[1].axhline(0.5, color="gray", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/tuning_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to results/tuning_results.png")

## 6. Full Training

Using best hyperparameters from tuning.

**6 total runs:** 3 varieties × 2 seeds

In [ ]:
# Cell 12 — train all adapters with best hyperparameters

all_train_paths = {}  

for variety in VARIETIES:
    print(f"\n{'='*60}")
    print(f"Training adapter: {variety}")
    print(f"{'='*60}")
    all_train_paths[variety] = {}

    train_split = ds["train"].filter(lambda x: x["variety"] == variety)
    val_split   = ds["validation"].filter(lambda x: x["variety"] == variety)

    print(f"  train: {len(train_split)} | val: {len(val_split)}")

    # class weights specific to this variety's imbalance ratio
    weights = class_weights(train_split, TASK)

    train_tok = tokenize_dataset(train_split, tokenizer, TASK, MAX_LENGTH)
    val_tok   = tokenize_dataset(val_split,   tokenizer, TASK, MAX_LENGTH)

    for seed in SEEDS:
        print(f"\n  Seed {seed}")

        model = apply_lora(base_model, LoRAConfig(r=BEST_R, lora_alpha=BEST_R*2))

        run_dir = os.path.join(ADAPTER_DIR, f"{variety.replace('-','_')}_seed{seed}")

        args = training_args(
            output_dir = run_dir,
            variety    = variety,
            seed       = seed,
            epochs     = 3,
            batch_size = 8,
            lr         = BEST_LR,
        )

        WeightedTrainer = new_weighted_class(weights)
        trainer = WeightedTrainer(
            model         = model,
            args          = args,
            train_dataset = train_tok,
            eval_dataset  = val_tok,
        )

        trainer.train()
        model.save_pretrained(run_dir)
        all_train_paths[variety][seed] = run_dir
        print(f"  Saved: {run_dir}")

print("\nAll adapters trained successfully")

In [ ]:
# Cell 13 — upload all adapters to HuggingFace Hub

for variety in VARIETIES:
    # upload seed=42 as canonical adapter
    path  = all_train_paths[variety][42]
    model = PeftModel.from_pretrained(base_model, path)
    push_adapter_to_hub(model, tokenizer, variety, MODEL_KEY)
    print()

print("All adapters on HuggingFace Hub")

## 7. Cross-Variety Evaluation

For each adapter: predict on all 3 test sets, both seeds.

In [ ]:
# Cell 14 — cross-variety evaluation

def predict(adapter_path, base_model, tokenizer, test_ds, max_len=128):
    #Load adapter and run batch inference on test dataset
    peft_model = PeftModel.from_pretrained(base_model, adapter_path)
    peft_model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    peft_model = peft_model.to(device)

    y_true, y_pred = [], []
    texts  = list(test_ds["text"])
    labels = list(test_ds[TASK])

    for i in range(0, len(texts), 16):
        batch = tokenizer(
            texts[i:i+16],
            truncation     = True,
            padding        = "max_length",
            max_length     = max_len,
            return_tensors = "pt",
        ).to(device)

        with torch.no_grad():
            preds = peft_model(**batch).logits.argmax(-1).cpu().tolist()

        y_true.extend([int(l) for l in labels[i:i+16]])
        y_pred.extend(preds)

    return y_true, y_pred


# all_eval[train_variety][test_variety][seed]
all_eval = {v: {v2: {} for v2 in VARIETIES} for v in VARIETIES}

for train_var in VARIETIES:
    print(f"\n{'='*55}")
    print(f"Evaluating: {train_var} adapter")
    print(f"{'='*55}")

    for test_var in VARIETIES:
        test_split = ds["test"].filter(lambda x: x["variety"] == test_var)
        print(f"\n  Testing on {test_var} ({len(test_split)} examples)")

        for seed in SEEDS:
            path   = all_train_paths[train_var][seed]
            y_true, y_pred = predict(path, base_model, tokenizer, test_split)

            macro_f1  = f1_score(y_true, y_pred, average="macro")
            precision = precision_score(y_true, y_pred, average="macro")
            recall    = recall_score(y_true, y_pred, average="macro")
            per_class = f1_score(y_true, y_pred, average=None)
            report    = classification_report(
                y_true, y_pred,
                target_names=["Not Sarcastic", "Sarcastic"],
                digits=4,
            )

            # check if model predicts only one class
            if len(set(y_pred)) == 1:
                print(f"    WARNING seed={seed}: model predicts only class {set(y_pred)}")
                print(f"    Class imbalance not handled — check WeightedTrainer")

            print(f"    seed={seed} | Macro-F1={macro_f1:.4f} | "
                  f"Precision={precision:.4f} | Recall={recall:.4f} | "
                  f"Sarcasm-F1={per_class[1]:.4f}")

            all_eval[train_var][test_var][seed] = {
                "y_true"   : y_true,
                "y_pred"   : y_pred,
                "macro_f1" : round(macro_f1,      4),
                "precision": round(precision,      4),
                "recall"   : round(recall,         4),
                "f1_class0": round(per_class[0],   4),
                "f1_class1": round(per_class[1],   4),
                "report"   : report,
            }

print("\nEvaluation complete")

## 8. Average Across Seeds

In [ ]:
# Cell 15 — average run1 (seed=42) and run2 (seed=123)

avg = {v: {v2: {} for v2 in VARIETIES} for v in VARIETIES}

print("Averaged results (mean ± std over 2 seeds)\n")

for tv in VARIETIES:
    for tv2 in VARIETIES:
        r1 = all_eval[tv][tv2][42]
        r2 = all_eval[tv][tv2][123]

        for metric in ["macro_f1", "precision", "recall", "f1_class0", "f1_class1"]:
            v1, v2_ = r1[metric], r2[metric]
            avg[tv][tv2][f"{metric}_mean"] = round((v1 + v2_) / 2, 4)
            avg[tv][tv2][f"{metric}_std"]  = round(abs(v1 - v2_) / 2, 4)

        m = avg[tv][tv2]["macro_f1_mean"]
        s = avg[tv][tv2]["macro_f1_std"]
        print(f"  {tv} → {tv2}: Macro-F1 = {m:.4f} ± {s:.4f}")

In [ ]:
# Cell 16 — build 3x3 result matrices

def build_matrix(avg, metric="macro_f1_mean"):
    matrix = pd.DataFrame(index=VARIETIES, columns=VARIETIES, dtype=float)
    for tv in VARIETIES:
        for tv2 in VARIETIES:
            matrix.loc[tv, tv2] = avg[tv][tv2][metric]
    matrix.index.name   = "Trained on"
    matrix.columns.name = "Tested on"
    return matrix

matrix_macro = build_matrix(avg, "macro_f1_mean")
matrix_sarc  = build_matrix(avg, "f1_class1_mean")
matrix_prec  = build_matrix(avg, "precision_mean")
matrix_rec   = build_matrix(avg, "recall_mean")

print("=== Macro-F1 Matrix ===")
print(matrix_macro.to_string(float_format="{:.4f}".format))

print("\n=== Sarcasm Class F1 Matrix ===")
print(matrix_sarc.to_string(float_format="{:.4f}".format))

## 9. Visualisation

In [ ]:
# Cell 17 — cross-variety heatmap matrices

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, matrix, title in zip(
    axes,
    [matrix_macro, matrix_sarc],
    ["Macro-F1 (primary metric)", "Sarcasm Class F1"],
):
    sns.heatmap(
        matrix.astype(float),
        annot      = True,
        fmt        = ".4f",
        cmap       = "YlOrRd",
        vmin       = 0.0,
        vmax       = 1.0,
        linewidths = 0.5,
        ax         = ax,
        annot_kws  = {"size": 11, "weight": "bold"},
    )
    ax.set_title(
        f"Cross-variety {title}\nLoRA Adapters ({MODEL_KEY}) — mean over 2 seeds",
        fontsize=11, fontweight="bold"
    )
    ax.set_xlabel("Tested on", fontsize=10)
    ax.set_ylabel("Trained on", fontsize=10)

    for i in range(len(VARIETIES)):
        ax.add_patch(plt.Rectangle(
            (i, i), 1, 1, fill=False, edgecolor="blue", lw=2.5
        ))

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/matrices.png")

In [ ]:
# Cell 18 — confusion matrices for in-variety diagonal

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(
    f"Confusion Matrices — In-variety Performance (seed=42)\n"
    f"Task: {TASK} | Model: {MODEL_KEY}",
    fontsize=13, fontweight="bold"
)

for ax, variety in zip(axes, VARIETIES):
    data = all_eval[variety][variety][42]
    cm   = confusion_matrix(data["y_true"], data["y_pred"])
    cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    sns.heatmap(
        cm_n,
        annot      = True,
        fmt        = ".2%",
        cmap       = "Blues",
        xticklabels= ["Not Sarc.", "Sarcastic"],
        yticklabels= ["Not Sarc.", "Sarcastic"],
        ax         = ax,
        cbar       = False,
    )
    # add raw counts below percentages
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j+0.5, i+0.7, f"n={cm[i,j]}",
                    ha="center", va="center", fontsize=8, color="gray")

    f1 = all_eval[variety][variety][42]["macro_f1"]
    sc = all_eval[variety][variety][42]["f1_class1"]
    ax.set_title(f"{variety}\nMacro-F1={f1:.4f} | Sarcasm-F1={sc:.4f}",
                 fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/confusion_matrices.png")

In [ ]:
# Cell 19 — precision/recall matrices

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, matrix, title in zip(
    axes,
    [matrix_prec, matrix_rec],
    ["Macro Precision", "Macro Recall"],
):
    sns.heatmap(
        matrix.astype(float),
        annot=True, fmt=".4f", cmap="Blues",
        vmin=0.0, vmax=1.0, linewidths=0.5, ax=ax,
        annot_kws={"size": 11},
    )
    ax.set_title(f"Cross-variety {title}\n{MODEL_KEY}", fontsize=11)
    ax.set_xlabel("Tested on")
    ax.set_ylabel("Trained on")
    for i in range(len(VARIETIES)):
        ax.add_patch(plt.Rectangle((i,i),1,1,fill=False,edgecolor="blue",lw=2.5))

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/precision_recall.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 20 — full classification reports for in-variety runs

print("=== Classification Reports — In-variety (seed=42) ===\n")
for variety in VARIETIES:
    print(f"{'─'*50}")
    print(f"{variety} adapter on {variety} test set")
    print(f"{'─'*50}")
    print(all_eval[variety][variety][42]["report"])

## 10.Export Results

In [ ]:
# Cell 21 — save all results to CSV

rows = []
for tv in VARIETIES:
    for tv2 in VARIETIES:
        r = avg[tv][tv2]
        rows.append({
            "Trained on"      : tv,
            "Tested on"       : tv2,
            "Macro-F1 mean"   : r["macro_f1_mean"],
            "Macro-F1 std"    : r["macro_f1_std"],
            "Precision mean"  : r["precision_mean"],
            "Recall mean"     : r["recall_mean"],
            "Sarcasm-F1 mean" : r["f1_class1_mean"],
            "Sarcasm-F1 std"  : r["f1_class1_std"],
        })

df = pd.DataFrame(rows)
df.to_csv(f"{RESULTS_DIR}/results.csv", index=False)
print("Saved: results/results.csv")
print(df.to_string(index=False))

In [ ]:
# Cell 22 — save tuning results

tune_df.to_csv(f"{RESULTS_DIR}/tuning_results.csv", index=False)
print("Saved: results/tuning_results.csv")

In [ ]:
# Cell 23 — download all result files 
from google.colab import files

for fname in [
    "results/matrices.png",
    "results/confusion_matrices.png",
    "results/precision_recall.png",
    "results/tuning_results.png",
    "results/results.csv",
    "results/tuning_results.csv",
]:
    if os.path.exists(fname):
        files.download(fname)
        print(f"Downloaded: {fname}")

print("\nAll files downloaded")